# 06 — Classification + 07 Transit Fitting
Apply the trained classifier to all science stars, then fit transit parameters
for any planet candidates found.

## 1. Imports

In [1]:
pip install "PyYAML>=6.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install batman-package

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Run this in a notebook cell
import batman
params = batman.TransitParams()
params.t0 = 0
params.per = 1.0
params.rp = 0.1
params.a = 15.0
params.inc = 87.0
params.ecc = 0.0
params.w = 90.0
params.u = [0.1, 0.3]
params.limb_dark = 'quadratic'
print('batman working correctly!')
print(f'batman version: {batman.__version__}')

batman working correctly!
batman version: 2.5.1


In [4]:
import astropy
import yaml
print(f'astropy: {astropy.__version__}')
print(f'PyYAML: {yaml.__version__}')

astropy: 8.0.0
PyYAML: 6.0.3


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, pickle, warnings
warnings.filterwarnings('ignore')

from scipy.optimize import curve_fit
from scipy import stats

try:
    import batman
    BATMAN_OK = True
    print('batman available ✅')
except ImportError:
    BATMAN_OK = False
    print('batman not found — using simple trapezoid model instead')

print('Imports OK!')

batman available ✅
Imports OK!


## 2. Load Everything

In [6]:
# Corrupted stars — exclude from all analysis
EXCLUDE_TICS = ['261203535']

In [13]:
import pickle
import os

MODELS_DIR = '../models/'

# Check all model files exist first
required = ['random_forest.pkl', 'xgboost_model.pkl',
            'label_encoder.pkl', 'feature_cols.pkl']

missing = [f for f in required if not os.path.exists(os.path.join(MODELS_DIR, f))]

if missing:
    print(f'❌ Missing model files: {missing}')
    print('Re-run notebook 05 first to train and save models')
    ML_AVAILABLE = False
else:
    with open(os.path.join(MODELS_DIR, 'random_forest.pkl'),  'rb') as f:
        rf_model  = pickle.load(f)
    with open(os.path.join(MODELS_DIR, 'xgboost_model.pkl'), 'rb') as f:
        xgb_model = pickle.load(f)
    with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
        le        = pickle.load(f)
    with open(os.path.join(MODELS_DIR, 'feature_cols.pkl'),  'rb') as f:
        feat_cols = pickle.load(f)
    ML_AVAILABLE = True

    print('Models loaded ✅')
    print(f'Classes  : {le.classes_}')
    print(f'Features : {len(feat_cols)}')

Models loaded ✅
Classes  : ['eclipsing_binary' 'planet']
Features : 31


## 3. Apply Classifier to All Science Stars

In [14]:
if not ML_AVAILABLE:
    print('No models available — using rule_label from results.csv instead')
    features_df['ml_label']      = features_df.get('rule_label', 'unknown')
    features_df['ml_confidence'] = features_df.get('rule_confidence', 0.5)
else:
    # Check all feat_cols exist in features_df
    missing_cols = [c for c in feat_cols if c not in features_df.columns]
    if missing_cols:
        print(f'⚠️  Missing feature columns: {missing_cols}')
        print('Adding them as zeros')
        for c in missing_cols:
            features_df[c] = 0.0

    X_sci     = features_df[feat_cols].fillna(0).values

    # Ensemble — average RF and XGB probabilities
    rf_probs  = rf_model.predict_proba(X_sci)
    xgb_probs = xgb_model.predict_proba(X_sci)
    ens_probs = (rf_probs + xgb_probs) / 2.0

    ens_labels = le.inverse_transform(np.argmax(ens_probs, axis=1))
    ens_conf   = ens_probs.max(axis=1)

    features_df['ml_label']      = ens_labels
    features_df['ml_confidence'] = ens_conf

    print('Ensemble classification (RF + XGBoost):')
    print(features_df['ml_label'].value_counts())
    print()
    print(features_df[['tic_id','ml_label','ml_confidence']].to_string())

Ensemble classification (RF + XGBoost):
ml_label
planet              11
eclipsing_binary     3
Name: count, dtype: int64

       tic_id          ml_label  ml_confidence
0   149603524  eclipsing_binary       0.584648
1   229742722            planet       0.707221
2   237201858            planet       0.677221
3   260647166  eclipsing_binary       0.665781
4   261136641            planet       0.803894
5   261136679            planet       0.766394
6   261136765            planet       0.558155
7   261139167            planet       0.734913
8   261155555            planet       0.540318
9   271893367            planet       0.597340
10  279741379  eclipsing_binary       0.575284
11  350618622            planet       0.740115
12  441075486            planet       0.760821
13   55525572            planet       0.576265


## 4. Save Classification Results

In [ ]:
import pandas as pd
import os

features_df = pd.read_csv('../outputs/features.csv')
bls_df      = pd.read_csv('../outputs/bls_all_results.csv')

features_df['tic_id'] = features_df['tic_id'].astype(str).str.replace('.0', '', regex=False)
bls_df['tic_id']      = bls_df['tic_id'].astype(str).str.replace('.0', '', regex=False)

EXCLUDE_TICS = ['261203535']
features_df  = features_df[~features_df['tic_id'].isin(EXCLUDE_TICS)].reset_index(drop=True)
bls_df       = bls_df[~bls_df['tic_id'].isin(EXCLUDE_TICS)].reset_index(drop=True)

if 'ml_label' not in features_df.columns:
    if os.path.exists('../outputs/results.csv'):
        results_df = pd.read_csv('../outputs/results.csv')
        results_df['tic_id'] = results_df['tic_id'].astype(str).str.replace('.0', '', regex=False)
        features_df = features_df.merge(
            results_df[['tic_id', 'ml_label', 'ml_confidence']],
            on='tic_id', how='left'
        )
    else:
        features_df['ml_label']      = 'unknown'
        features_df['ml_confidence'] = 0.5

bls_cols = ['tic_id', 'period_days', 'duration_days', 'depth_ppm', 'snr', 'transit_time']
bls_cols = [c for c in bls_cols if c in bls_df.columns]
final_df  = features_df.merge(bls_df[bls_cols], on='tic_id', suffixes=('', '_bls'))

# Fill missing primary cols from BLS fallback then drop ALL _bls suffix columns
for col in ['period_days', 'duration_days', 'depth_ppm', 'snr']:
    bls_col = f'{col}_bls'
    if bls_col in final_df.columns:
        final_df[col] = final_df[col].fillna(final_df[bls_col])
        final_df.drop(columns=[bls_col], inplace=True)

# Drop any remaining _bls columns
bls_leftovers = [c for c in final_df.columns if c.endswith('_bls')]
if bls_leftovers:
    final_df.drop(columns=bls_leftovers, inplace=True)

print(f'final_df: {final_df.shape}')
print(final_df[['tic_id', 'ml_label', 'ml_confidence']].to_string())

In [17]:
final_df.to_csv('../outputs/final_classification.csv', index=False)
print('Saved to outputs/final_classification.csv')

# Only use columns that exist
available = ['tic_id','ml_label','ml_confidence','period_days','depth_ppm','snr']
available = [c for c in available if c in final_df.columns]

summary = final_df[available].copy()

# Rename only existing columns
rename_map = {
    'tic_id'         : 'TIC ID',
    'ml_label'       : 'Label',
    'ml_confidence'  : 'Confidence',
    'period_days'    : 'Period (days)',
    'depth_ppm'      : 'Depth (ppm)',
    'snr'            : 'SNR'
}
summary.rename(columns={k:v for k,v in rename_map.items()
                         if k in summary.columns}, inplace=True)

if 'Confidence' in summary.columns:
    summary['Confidence'] = summary['Confidence'].map('{:.1%}'.format)
if 'Depth (ppm)' in summary.columns:
    summary['Depth (ppm)'] = summary['Depth (ppm)'].apply(
        lambda x: f'{float(x):.0f}' if pd.notna(x) else 'N/A'
    )
if 'SNR' in summary.columns:
    summary['SNR'] = summary['SNR'].apply(
        lambda x: f'{float(x):.2f}' if pd.notna(x) else 'N/A'
    )

print()
print(summary.to_string(index=False))

Saved to outputs/final_classification.csv

   TIC ID            Label Confidence  Period (days) Depth (ppm)   SNR
149603524 eclipsing_binary      58.5%       8.824835      999225  6.50
229742722           planet      70.7%       1.050601      999752  4.43
237201858           planet      67.7%       4.845005      999581  4.61
260647166 eclipsing_binary      66.6%       7.933500      999656  8.41
261136641           planet      80.4%       1.089596      999882  5.00
261136679           planet      76.6%       6.267548      999810 28.25
261136765           planet      55.8%       3.282632      999499  4.98
261139167           planet      73.5%       1.751655      999794  5.01
261155555           planet      54.0%       4.631142      996914 35.55
271893367           planet      59.7%      10.629036      999280 11.33
279741379 eclipsing_binary      57.5%      11.843468      999396 16.05
350618622           planet      74.0%       0.735783      999899  3.82
441075486           planet      76

## 5. Transit Fitting for Planet Candidates
For stars classified as planets, fit a transit model to get precise parameters.

In [18]:
def trapezoid_transit(phase, depth, t_ingress, t_flat):
    """
    Simple trapezoid transit model.
    Used when batman is not available.
    phase     : array of phase values centered at 0
    depth     : transit depth (fraction)
    t_ingress : ingress/egress duration (in phase units)
    t_flat    : flat bottom duration (in phase units)
    """
    flux = np.ones_like(phase)
    half_flat    = t_flat / 2.0
    half_total   = half_flat + t_ingress

    for i, p in enumerate(phase):
        ap = abs(p)
        if ap <= half_flat:
            flux[i] = 1.0 - depth
        elif ap <= half_total:
            flux[i] = 1.0 - depth * (half_total - ap) / t_ingress
    return flux


def fit_transit_batman(time, flux, period, t0, depth_init, duration):
    """Fit transit using batman model."""
    import batman
    params         = batman.TransitParams()
    params.t0      = t0
    params.per     = period
    params.rp      = np.sqrt(max(depth_init, 1e-6))
    params.a       = 10.0
    params.inc     = 89.0
    params.ecc     = 0.0
    params.w       = 90.0
    params.u       = [0.3, 0.1]
    params.limb_dark = 'quadratic'

    m    = batman.TransitModel(params, time)
    flux_model = m.light_curve(params)
    return flux_model, params


def fit_transit_simple(phase, flux, depth_init, duration_phase):
    """Fit trapezoid model."""
    try:
        p0     = [depth_init, duration_phase/4, duration_phase/2]
        bounds = ([0, 0, 0], [1, 0.5, 0.5])
        popt, pcov = curve_fit(trapezoid_transit, phase, flux,
                               p0=p0, bounds=bounds, maxfev=5000)
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception as e:
        return None, None

print('Transit fitting functions ready!')

Transit fitting functions ready!


## 6. Fit & Plot ALL Stars (with classification labels)

In [ ]:
transit_params_list = []

label_colors = {
    'planet'           : 'green',
    'planet_candidate' : 'limegreen',
    'eclipsing_binary' : 'red',
    'false_positive'   : 'darkorange',
    'noise'            : 'gray',
    'unknown'          : 'silver',
}

n_stars = len(final_df)
ncols   = 3
nrows   = int(np.ceil(n_stars / ncols))

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(18, 5 * nrows))
axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

for idx, (_, row) in enumerate(final_df.iterrows()):
    ax     = axes[idx]
    tic_id = str(row['tic_id']).replace('.0','')
    label  = row['ml_label']
    conf   = row['ml_confidence']
    period = row.get('period_days', row.get('period_days_bls', 1.0))
    t0     = row['transit_time']
    depth  = row['depth_ppm'] / 1e6
    dur    = row.get('duration_days', row.get('duration_days_bls', 0.1))
    color  = label_colors.get(label, 'steelblue')

    try:
        df_lc  = pd.read_csv(os.path.join(PROCESSED_DIR, f'TIC_{tic_id}.csv'))
        time   = df_lc['time'].values
        flux   = df_lc['flux'].values
        mask   = np.isfinite(time) & np.isfinite(flux)
        time   = time[mask]
        flux   = flux[mask]

        # Phase fold
        phase  = ((time - t0) % period) / period
        phase[phase > 0.5] -= 1.0
        sidx   = np.argsort(phase)
        phase  = phase[sidx]
        flux_s = flux[sidx]

        # Plot phase-folded
        ax.plot(phase, flux_s, '.', color='gray',
                markersize=1.5, alpha=0.4, label='Data')

        # Bin the folded curve for clarity
        n_bins  = 50
        bins    = np.linspace(-0.5, 0.5, n_bins+1)
        bin_mid = (bins[:-1] + bins[1:]) / 2
        bin_flux= [np.median(flux_s[(phase>=bins[i]) & (phase<bins[i+1])])
                   if np.any((phase>=bins[i]) & (phase<bins[i+1]))
                   else np.nan
                   for i in range(n_bins)]
        ax.plot(bin_mid, bin_flux, 'o-', color=color,
                markersize=4, lw=1.5, label='Binned', zorder=5)

        # Fit transit model if planet candidate
        fit_label = ''
        if label in ['planet', 'planet_candidate'] and depth > 0:
            dur_phase = dur / period
            transit_mask = np.abs(phase) < dur_phase
            if transit_mask.sum() > 5:
                popt, perr = fit_transit_simple(phase, flux_s, depth, dur_phase)
                if popt is not None:
                    phase_fine  = np.linspace(-0.5, 0.5, 500)
                    flux_model  = trapezoid_transit(phase_fine, *popt)
                    ax.plot(phase_fine, flux_model, 'b-', lw=2,
                            label='Transit model', zorder=6)
                    fitted_depth    = popt[0]
                    fitted_dur_days = (popt[1]*2 + popt[2]) * period
                    fit_label = f'Fit depth={fitted_depth*1e6:.0f}ppm'

                    transit_params_list.append({
                        'tic_id'          : tic_id,
                        'label'           : label,
                        'period_days'     : period,
                        'depth_ppm_fit'   : fitted_depth * 1e6,
                        'duration_days_fit': fitted_dur_days,
                        'depth_err_ppm'   : (perr[0] if perr is not None else np.nan) * 1e6,
                        'confidence'      : conf
                    })

        ax.set_xlim(-0.5, 0.5)
        ax.set_xlabel('Phase', fontsize=8)
        ax.set_ylabel('Norm. Flux', fontsize=8)
        ax.set_title(
            f'TIC {tic_id}\n'
            f'{label.upper()}  ({conf:.0%})\n'
            f'P={period:.2f}d  SNR={row["snr"]:.1f}  {fit_label}',
            fontsize=8, color=color
        )
        ax.legend(fontsize=6)

    except Exception as e:
        ax.text(0.5, 0.5, f'TIC {tic_id}\nError: {str(e)[:40]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=7)

# Hide empty subplots
for idx in range(len(final_df), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Phase-Folded Light Curves — All Science Targets\n'
             '(Green=Planet, Limegreen=Candidate, Red=EB, Orange=FP, Gray=Noise)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print('All light curves plotted!')

## 7. Save Transit Parameters

In [20]:
if transit_params_list:
    tp_df = pd.DataFrame(transit_params_list)
    tp_df.to_csv('../outputs/transit_params.csv', index=False)
    print('Transit parameters for planet candidates:')
    print(tp_df.to_string(index=False))
else:
    print('No planet candidates found with fittable transits.')
    print('This is expected if all stars have SNR < 7.')
    print()
    # Save BLS parameters as best estimates instead
    bls_params = final_df[['tic_id','ml_label','ml_confidence',
                            'period_days','depth_ppm','snr']].copy()
    bls_params['duration_hours'] = final_df['duration_hours']
    bls_params['note'] = 'BLS estimate (no transit fit — SNR too low)'
    bls_params.to_csv('../outputs/transit_params.csv', index=False)
    print('BLS parameter estimates saved to transit_params.csv')
    print(bls_params[['tic_id','ml_label','period_days',
                       'depth_ppm','duration_hours']].to_string(index=False))

Transit parameters for planet candidates:
   tic_id  label  period_days  depth_ppm_fit  duration_days_fit  depth_err_ppm  confidence
229742722 planet     1.050601     550.036243           0.010152     452.253291    0.707221
237201858 planet     4.845005     463.800223           0.056143     162.844835    0.677221
261136641 planet     1.089596     236.003271           0.008779     141.140189    0.803894
261136679 planet     6.267548     180.657719           0.123594       6.257083    0.766394
261136765 planet     3.282632     545.382946           0.039274     125.404207    0.558155
261139167 planet     1.751655     199.088177           0.071534      45.029185    0.734913
261155555 planet     4.631142    3252.591617           0.146163     111.419234    0.540318
271893367 planet    10.629036     708.313477           1.257564      95.413693    0.597340
350618622 planet     0.735783     170.155689           0.013340     180.770364    0.740115
441075486 planet     1.120751    4426.712475    

## 8. Final Summary Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Label pie chart
label_counts = final_df['ml_label'].value_counts()
pie_colors   = [label_colors.get(l, 'steelblue') for l in label_counts.index]
axes[0].pie(label_counts.values, labels=label_counts.index,
            colors=pie_colors, autopct='%1.0f%%',
            startangle=90, textprops={'fontsize':10})
axes[0].set_title('Classification Breakdown', fontsize=12)

# Period vs Depth scatter — use .get() so any new class gets a neutral colour
sc = axes[1].scatter(
    final_df['period_days'],
    final_df['depth_ppm'],
    c=[label_colors.get(l, 'steelblue') for l in final_df['ml_label']],
    s=final_df['snr'].clip(1,100) * 5,
    edgecolors='k', linewidths=0.5, alpha=0.8
)
axes[1].set_xlabel('Period (days)')
axes[1].set_ylabel('Transit Depth (ppm)')
axes[1].set_title('Period vs Depth\n(size = SNR)', fontsize=12)
axes[1].set_yscale('log')

# Confidence histogram
axes[2].hist(final_df['ml_confidence'] * 100, bins=10,
             color='steelblue', edgecolor='white')
axes[2].set_xlabel('Confidence (%)')
axes[2].set_ylabel('Count')
axes[2].set_title('Classification Confidence\nDistribution', fontsize=12)

plt.suptitle('Exoplanet Detection Pipeline — Quick Summary', fontsize=14)
plt.tight_layout()
plt.show()
print('Quick-look summary plotted (report-quality figures in 08_visualization.ipynb)')

---
## ✅ Pipeline Complete!
**All outputs saved:**
- `outputs/final_classification.csv` — label + confidence for all stars
- `outputs/transit_params.csv` — period, depth, duration estimates
- `outputs/plots/all_classified_lightcurves.png` — phase-folded plots
- `outputs/plots/final_summary.png` — summary charts

**Next Step → Run `08_visualization.ipynb` for final report-quality plots**